In [ ]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB
import os
from time import time
from flask import Flask, request
from json import dumps
import json
from typing import Tuple, List, Dict

Tenant_Min = Dict[str, gp.Var]
Tenant_Consumed = Dict[str, gp.Var]

previous_w: Dict[str,float] = {}

def run_model(_host_cap, _t0, _t1, _t2):

    # MIP  model formulation
    m = gp.Model("lb")

    host_cap = m.addVar(lb=_host_cap, ub=_host_cap, vtype=GRB.CONTINUOUS,
                        name="host_cap")

    t0 = m.addVar(lb=_t0, ub=_t0, vtype=GRB.CONTINUOUS, name="t0")
    t00 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t00")
    t01 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t01")

    t1 = m.addVar(lb=_t1, ub=_t1, vtype=GRB.CONTINUOUS, name="t1")
    t11 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t11")
    t12 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t12")

    t2 = m.addVar(lb=_t2, ub=_t2, vtype=GRB.CONTINUOUS, name="t2")
    t20 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t20")

    # sp_cap_0 = m.addVar(lb=float("-inf"), vtype=GRB.CONTINUOUS, name="sp_cap_0")
    # sp_cap_1 = m.addVar(lb=float("-inf"), vtype=GRB.CONTINUOUS, name="sp_cap_1")
    # sp_cap_2 = m.addVar(lb=float("-inf"), vtype=GRB.CONTINUOUS, name="sp_cap_2")

    sp_cap_0 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_0")
    sp_cap_1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_1")
    sp_cap_2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_2")

    share0 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="share0")
    share1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="share1")
    share2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="share2")
    
    fshare0 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="fshare0")
    fshare1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="fshare1")
    fshare2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="fshare2")
    
    smallest_sp_cap_log = m.addVar(lb=float("-inf"), vtype=GRB.CONTINUOUS,
                                   name="smallest_sp_cap_log")

    smallest_sp_cap = m.addVar(lb=0, vtype=GRB.CONTINUOUS,
                               name="smallest_sp_cap")

    m.setObjective(smallest_sp_cap_log, GRB.MAXIMIZE)

    m.addGenConstrMin(smallest_sp_cap, [sp_cap_0, sp_cap_1, sp_cap_2],
                      name="min_sp_cap")
    
    m.addGenConstrLog(smallest_sp_cap, smallest_sp_cap_log)

    m.addConstr(t20 + t00 + sp_cap_0 <= host_cap, name="h0")
    m.addConstr(t01 + t11 + sp_cap_1 <= host_cap, name="h1")
    m.addConstr(t12 +       sp_cap_2 <= host_cap, name="h2")

    m.addConstr(t0 >= t00 + t01, name="t0")
    m.addConstr(t1 >= t11 + t12, name="t1")
    m.addConstr(t2 >= t20, name="t2")
    
    m.addConstr(fshare0 == 2 * host_cap, name="fshare_0")
    m.addConstr(fshare1 == 2.5 * host_cap, name="fshare_1")
    m.addConstr(fshare2 == 1 * host_cap, name="fshare_2")
    
    m.addGenConstrMin(share0, [fshare0, t0], name="share0")
    m.addGenConstrMin(share1, [fshare1, t1], name="share1")
    m.addGenConstrMin(share2, [fshare2, t2], name="share2")
    
    m.addConstr(share0 == t00 + t01, name="share_0")
    m.addConstr(share1 == t11 + t12, name="share_1")
    m.addConstr(share2 == t20, name="share_2")

    m.optimize()
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        to_return = {
            "status": m.Status,
            "t00": vars["t00"],
            "t01": vars["t01"],
            "t11": vars["t11"],
            "t12": vars["t12"],
            "t20": vars["t20"],
        }
        print(to_return)
        return to_return
    else:
        to_return = {
            "status": m.Status,
            "t00": 0.0,
            "t01": 0.0,
            "t11": 0.0,
            "t12": 0.0,
            "t20": 0.0,
        }
        print(to_return)
        return to_return

def run_new_model(_host_cap, _t0, _t1, _t2):

    # MIP  model formulation
    m = gp.Model("lb")

    host_cap = m.addVar(lb=_host_cap, ub=_host_cap, vtype=GRB.CONTINUOUS, name="host_cap")

    t_min = m.addVar(lb=0.0, ub=0.0, vtype=GRB.CONTINUOUS, name="t_min")

    t0 = m.addVar(lb=_t0, ub=_t0, vtype=GRB.CONTINUOUS, name="t0")
    t00 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t00")
    t01 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t01")

    t1 = m.addVar(lb=_t1, ub=_t1, vtype=GRB.CONTINUOUS, name="t1")
    t11 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t11")
    t12 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t12")

    t2 = m.addVar(lb=_t2, ub=_t2, vtype=GRB.CONTINUOUS, name="t2")
    t20 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="t20")
    
    h0_sum = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="h0_sum")
    h1_sum = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="h1_sum")
    h2_sum = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="h2_sum")

    h0_log = m.addVar(vtype=GRB.CONTINUOUS, name="h0_log")
    h1_log = m.addVar(vtype=GRB.CONTINUOUS, name="h1_log")
    h2_log = m.addVar(vtype=GRB.CONTINUOUS, name="h2_log")

    # sp_cap_0 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_0")
    # sp_cap_1 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_1")
    # sp_cap_2 = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS, name="sp_cap_2")

    # smallest_sp_cap = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
    #       name="smallest_sp_cap")

    # objective = (h0_log + h1_log + h2_log) + \
    #     ((host_cap - h0_sum) + (host_cap - h1_sum) + (host_cap - h2_sum))

    m.setObjective((h0_log + h1_log + h2_log) + \
        ((host_cap - h0_sum) + (host_cap - h1_sum) + (host_cap - h2_sum)), GRB.MAXIMIZE)

    # m.addGenConstrMin(smallest_sp_cap, [sp_cap_0, sp_cap_1, sp_cap_2],
    #       name="min_sp_cap")

    m.addConstr(t20 + t00 <= host_cap, name="h0")
    m.addConstr(t01 + t11 <= host_cap, name="h1")
    m.addConstr(t12       <= host_cap, name="h2")
    
    # add constraint for sum of each host
    m.addConstr(t20 + t00 == h0_sum, name="h0_sum")
    m.addConstr(t01 + t11 == h1_sum, name="h1_sum")
    m.addConstr(t12       == h2_sum, name="h2_sum")
    
    m.addGenConstrLog(h0_sum, h0_log)
    m.addGenConstrLog(h1_sum, h1_log)
    m.addGenConstrLog(h2_sum, h2_log)

    m.addConstr(t00 + t01 >= t0, name="t0")
    m.addConstr(t11 + t12 >= t1, name="t1")
    m.addConstr(t20       >= t2, name="t2")
    
    m.addConstr(t00 + t01 >= t_min, name="t0")
    m.addConstr(t11 + t12 >= t_min, name="t1")
    m.addConstr(t20       >= t_min, name="t2")
    
    m.optimize()
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        to_return = {
            "status": m.Status,
            "t00": vars["t00"],
            "t01": vars["t01"],
            "t11": vars["t11"],
            "t12": vars["t12"],
            "t20": vars["t20"],
        }
        # print(vars)
        return to_return
    else:
        to_return = {
            "status": m.Status,
            "t00": 0.0,
            "t01": 0.0,
            "t11": 0.0,
            "t12": 0.0,
            "t20": 0.0,
        }
        # print(vars)
        return to_return

def run_general_model(_host_cap: float, _t: List[float]) -> str:

    # MIP  model formulation
    m = gp.Model("lb")
    
    n_hosts = len(_t)
    n_tenants = len(_t)
    
    host_cap = m.addVar(lb=_host_cap, ub=_host_cap, vtype=GRB.CONTINUOUS,
                        name="host_cap")
    
    t = [m.addVar(lb=_t[i], ub=_t[i], vtype=GRB.CONTINUOUS, name=f"t{i}")
         for i in range(n_tenants)]

    w = {}
    
    for tenant in range(n_tenants):
        hosts = [tenant] if tenant == 0 else [tenant-1, tenant]
        for host in hosts:
            w[tenant, host] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                       name=f"w{tenant}{host}")
            
    print(w)
             
    sp = {}
    for host in range(n_hosts):
        sp[host] = m.addVar(vtype=GRB.CONTINUOUS, name=f"sp{host}")
        
    log_sp = {}
    for host in range(n_hosts):
        log_sp[host] = m.addVar(vtype=GRB.CONTINUOUS, name=f"log_sp{host}")
    
    share = {}
    for tenant in range(n_tenants):
        share[tenant] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                 name=f"share{tenant}")
        
    fshare = {}
    for tenant in range(n_tenants):
        fshare[tenant] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                 name=f"fshare{tenant}")
        
    smallest_log_sp_cap = m.addVar(vtype=GRB.CONTINUOUS,
                               name="smallest_sp_cap")
    
    m.setObjective(smallest_log_sp_cap, GRB.MAXIMIZE)
    
    m.addGenConstrMin(smallest_log_sp_cap,
                      [sp[host] for host in range(n_hosts)],
                      name="min_sp_cap")
    
    # for host in range(n_hosts):
    #     m.addGenConstrLog(sp[host], log_sp[host], name=f"log_sp{host}")
   
    for host in range(n_hosts):
        m.addConstr(gp.quicksum((w[tenant, host]
                                for tenant in (
                                    [host] if host == (n_hosts - 1) else [host, host+1])))
                    + sp[host] == host_cap,
                    name=f"h{host}")
    
    for tenant in range(n_tenants):
        m.addConstr(
            gp.quicksum(
                (w[tenant, host]
                for host in ([tenant] if tenant == 0 else [tenant, tenant-1]))) 
            <= t[tenant],
            name=f"t{tenant}")

    for tenant in range(n_tenants):
        if tenant == 0:
            m.addConstr(fshare[tenant] == 0.5 * host_cap, name=f"fshare{tenant}")
        elif tenant == n_tenants - 1:
            m.addConstr(fshare[tenant] == 1.5 * host_cap, name=f"fshare{tenant}")
        else:
            m.addConstr(fshare[tenant] == 1 * host_cap, name=f"fshare{tenant}")
            
    for tenant in range(n_tenants):
        m.addGenConstrMin(share[tenant], [fshare[tenant], t[tenant]],
                          name=f"share{tenant}")
        
    for tenant in range(n_tenants):
        m.addConstr(
            share[tenant] <= gp.quicksum(
                (w[tenant, host]
                    for host in ([tenant] if tenant == 0 else [tenant, tenant-1]))),
            name=f"share_{tenant}")
        
    m.optimize()
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        print(vars)
        
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        to_return = {
            "status": m.Status,
            "t00": vars["w10"] + 0.00001,
            "t01": vars["w11"] + 0.00001,
            "t11": vars["w21"] + 0.00001,
            "t12": vars["w22"] + 0.00001,
            "t20": vars["w00"] + 0.00001,
        }
        print(to_return)
        return to_return
    else:
        to_return = {
            "status": m.Status,
            "t00": 0.0 + 0.00001,
            "t01": 0.0 + 0.00001,
            "t11": 0.0 + 0.00001,
            "t12": 0.0 + 0.00001,
            "t20": 0.0 + 0.00001,
        }
        print(to_return)
        return to_return

# class Host:
#     def __init__(self, name: str, cap: float):
#         self.name: str = name
#         self.cap: str = cap

# class Tenant:
#     def __init__(self, name: str, load: float, fshare: float = 0.0):
#         self.name: str = name
#         self.load: float = load
#         self.fshareload: float = fshare

# class Worker:
#     def __init__(self, name: str, tenant: str, host: str):
#         self.name: str = name
#         self.tenant: str = tenant
#         self.host: str = host
        
N_WORKERS_EXPONENTIAL_DISTR_LAMBDA = 17
WORKER_LOAD_EXPONENTIAL_DISTR_LAMBDA = 0.7
HOST_CAPACITY = 1.0

class Host:
    def __init__(self, name: str, cap: float):
        self.name: str = name
        self.cap: str = cap
        self.worker_ids: List[int] = []
        
    def __str__(self):
        return f"{self.name}: cap={self.cap}, n_workers={len(self.worker_ids)}"
        
class Tenant:
    def __init__(self, name: str, load: float, fshare: float = 0.0):
        self.name: str = name
        self.load: float = load
        self.fshareload: float = fshare
        
    def __str__(self):
        return f"{self.name}: load={self.load}, fshareload={self.fshareload}"

class Worker:
    def __init__(self, name: str, tenant: str, host: str, tenant_id: int = -1):
        self.name: str = name
        self.tenant: str = tenant
        self.tenant_id: int = tenant_id
        self.host: str = host
        
    def __str__(self):
        return f"{self.name}: tenant={self.tenant}, host={self.host}"

def get_topology(n_hosts: int) -> Tuple[List[Host], List[Tenant], List[Worker]]:
    
    print(f"number of hosts: {n_hosts}")
    
    n_tenants = int(n_hosts * 0.65)
    print(f"number of tenants: {n_tenants}")    
    
    n_workers_per_ms = list(map(int, np.random.exponential(
        N_WORKERS_EXPONENTIAL_DISTR_LAMBDA, size=n_tenants)))
    print(f"number of workers: {np.sum(n_workers_per_ms)}")
    
    hosts = [Host(f"host{i}", 1.0) for i in range(n_hosts)]
    
    tenants = []
    workers = []
    worker_id = 0
    for i in range(n_tenants):
        
        tenant_load = np.sum(np.random.exponential(
            WORKER_LOAD_EXPONENTIAL_DISTR_LAMBDA, size=n_workers_per_ms[i]))
        
        tenant = Tenant(f"tenant{i}", tenant_load)
        tenants.append(tenant)
        
        for j in range(n_workers_per_ms[i]):
            
            host_idx = np.random.randint(0, n_hosts)
            host_name = f"host{host_idx}"
            
            worker = Worker(f"tenant{i}_{j}", tenant.name, host_name, i)
            workers.append(worker)
            
            hosts[host_idx].worker_ids.append(worker_id)
            
            worker_id += 1
    
    for host in hosts:
        
        fshare_of_each_worker = HOST_CAPACITY / len(host.worker_ids) if len(host.worker_ids) > 0 else 0.0
        
        for worker_id in host.worker_ids:
            worker = workers[worker_id]
            tenant = tenants[worker.tenant_id]
            tenant.fshareload += fshare_of_each_worker
        
def run_admission_control(
    _hosts: List[Host],
    _tenants: List[Tenant],
    _workers: List[Worker]):
    
    """
    Algorithm:
    
    (Greedy solution:)
    
    1. For each tenant, divide demand on the pods
    2. For each pod, what is more resources needed than fair share?, if it is in negative, add it to the spare capacity of the node
    3. If spare capacity available at the node, then divide it among the pods in a maxmin way
    4. If more resources are needed for a node, then try all pods, if they can be moved to another node, if any can be moved, move as much as possible until no more resources are needed in the node
    5. As spare capacity is available, divide it among tenants in a maxmin fair way
    6. Repeat until no more resources can be shed off from a node, or all load is satisfied
    
    Constraint:
    1. 
    
    Now, with admission control, first we calculate demands and excess resource needs:
    App3: demand=1, fair share=1, excess needed=0
    App1: demand=3, fair share=2, excess needed=-1
    App2: demand=2, fair share=3, excess needed=1

    Now, how do I write the next step to determine that we allocate 1 node to allocate to app1-1?
    
    """
    
    pass
      
def rerun_generic_linear_model(
    m: gp.Model,
    t_min: Tenant_Min,
    t_consumed: Tenant_Consumed,
    _hosts: List[Host],
    _tenants: List[Tenant],
    _workers: List[Worker]) -> Tuple[str, gp.Model, Tenant_Min, Tenant_Consumed]:
    
    global previous_w
    
    # confirm if the topology is the same, if not, rerun the optimization from scratch
    was_previous_the_same_topology = all(worker.name in previous_w for worker in _workers) and len(previous_w) == len(_workers)
    if not was_previous_the_same_topology:
        
        raise Exception("The topology has changed, please rerun the optimization from scratch")
        
        return run_generic_linear_model(_hosts, _tenants, _workers)

    #  ============================= Modify Variables =============================
    
    for tenant in _tenants:
        t_min_value = min(tenant.fshareload, tenant.load)
        t_min[tenant.name].LB = t_min_value
        t_min[tenant.name].UB = t_min_value
        
    for tenant in _tenants:
        print(f"{tenant.name}: (lb: {min(tenant.fshareload, tenant.load)}, ub: {tenant.load})")
        t_consumed[tenant.name].LB = min(tenant.fshareload, tenant.load)
        t_consumed[tenant.name].UB = tenant.load
        
    # ============================ Update Model =============================
    
    m.update()
    
    # ============================== Optimize! =================================
    
    m.optimize()
    
    # =========================== Done Optimization ============================
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        print(vars)
        
    if m.Status == GRB.OPTIMAL:        
        
        vars = {v.varName: v.x for v in m.getVars()}
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
            else:
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        # set the previous weights to the current weights
        previous_w = {worker.name: vars[f"w_{worker.name}"] for worker in _workers}
        print("New previous weights:", previous_w)            
        
        print(to_return)
        
        return to_return, m, t_min, t_consumed

    else:
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = 0.0
            else:
                results[worker.tenant][worker.name] = 0.0
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        print(to_return)
        
        return to_return, m, t_min, t_consumed


def run_generic_linear_model(
    _hosts: List[Host],
    _tenants: List[Tenant],
    _workers: List[Worker]) -> Tuple[str, gp.Model, Tenant_Min, Tenant_Consumed]:
    
    global previous_w
    
    # =========================== Begin Optimization ===========================
    
    # MIP  model formulation
    m = gp.Model("lb")
    
    #  ============================= Set Variables =============================
    
    # set host capacity for each host
    cap = {}
    for h in _hosts:
        cap[h.name] = m.addVar(lb=h.cap, ub=h.cap, vtype=GRB.CONTINUOUS,
                        name=f"cap_{h.name}")
    
    # # set variables for the tenant loads
    # t = {}
    # for tenant in _tenants:
    #     t[tenant.name] = m.addVar(lb=tenant.load, ub=tenant.load, 
    #                               vtype=GRB.CONTINUOUS, name=f"t_{tenant.name}")
    
    # set variables for the workers
    w = {}   
    for worker in _workers:
        w[worker.name] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                           name=f"w_{worker.name}")
    
    print(w)

    t_min = {}
    for tenant in _tenants:
        t_min_value = min(tenant.fshareload, tenant.load)
        t_min[tenant.name] = m.addVar(lb=t_min_value, ub=t_min_value,
                                      vtype=GRB.CONTINUOUS,
                                 name=f"t_min_{tenant.name}")
        
    t_consumed = {}
    for tenant in _tenants:
        print(f"{tenant.name}: (lb: {min(tenant.fshareload, tenant.load)}, ub: {tenant.load})")
        # t_consumed[tenant.name] = m.addVar(lb=0.0,
        #                                    vtype=GRB.CONTINUOUS,
        #                          name=f"t_consumed_{tenant.name}")
        # We could also add the constraints here instead of adding them later, maybe that saves time in optimization
        t_consumed[tenant.name] = m.addVar(lb=min(tenant.fshareload, tenant.load), 
                                           ub=tenant.load,
                                           vtype=GRB.CONTINUOUS,
                                 name=f"t_consumed_{tenant.name}")
    
    t_excess_consumed = {}
    for tenant in _tenants:
        t_excess_consumed[tenant.name] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                 name=f"t_excess_consumed_{tenant.name}")
    
    t_log_excess_consumed = {}
    for tenant in _tenants:
        t_log_excess_consumed[tenant.name] = m.addVar(vtype=GRB.CONTINUOUS,
                                                      lb=-GRB.INFINITY,
                                                      ub=GRB.INFINITY,
                                 name=f"t_log_excess_consumed_{tenant.name}")
    
    one = m.addVar(lb=1.0, ub=1.0, vtype=GRB.CONTINUOUS, name="one")
    
    # ======================= Set Utilization Objective ========================
    
    sum_fshareloads = sum(tenant.fshareload for tenant in _tenants)
    
    weighted_sum_of_t_excess_consumption = gp.quicksum(
        ((tenant.fshareload / sum_fshareloads) * t_log_excess_consumed[tenant.name]
         for tenant in _tenants))
    
    # sum_workers = gp.quicksum((w[worker.name] for worker in _workers))
         
    m.setObjectiveN(-weighted_sum_of_t_excess_consumption, index=0, priority=2)
    
    # ============================ Set Constraints =============================
    
    # for each tenant, set t_log_excess_consumed = log(t_consumed)
    for tenant in _tenants:
        m.addGenConstrLog(t_excess_consumed[tenant.name],
                          t_log_excess_consumed[tenant.name])
    
    # for each tenant, set t_excess_consumed = t_consumed - t_min
    for tenant in _tenants:
        m.addConstr(
            t_excess_consumed[tenant.name] == 
            t_consumed[tenant.name] - t_min[tenant.name],
            name=f"t_excess_consumed_{tenant.name}")
    
    # at each h, sum(w ∈ h) <= cap
    for host in _hosts:
        m.addConstr(gp.quicksum(
            (w[worker.name] for worker in _workers if worker.host == host.name)) 
                    <= cap[host.name],
                    name=f"h_{host.name}")
        
    # for each tenant t, set t_consumed = sum(w ∈ t)
    for tenant in _tenants:
        m.addConstr(
            t_consumed[tenant.name] == gp.quicksum(
                (w[worker.name]
                for worker in _workers if worker.tenant == tenant.name)),
            name=f"t_consumed_{tenant.name}")
        
    # # at each t, t_consumed <= t
    # for tenant in _tenants:
    #     m.addConstr(
    #         t_consumed[tenant.name] <= tenant.load,
    #         name=f"t_upper_{tenant.name}")
    
    # # for each tenant, t_consumed >= t_min
    # for tenant in _tenants:
    #     m.addConstr(
    #         t_consumed[tenant.name] >= min(tenant.fshareload, tenant.load),
    #         name=f"t_lower_{tenant.name}")
        
    # ========================== Variance Objective ============================
    
    host_utilizations = []
    for host in _hosts:
        host_utilizations.append(gp.quicksum((w[worker.name] for worker in _workers if worker.host == host.name)))
    
    n = len(host_utilizations)
    
    # Auxiliary variables for absolute differences
    differences = {}
    for i in range(n):
        for j in range(i + 1, n):  # Only consider each pair once (i < j)
            differences[i, j] = m.addVar(vtype=GRB.CONTINUOUS, name=f"d_{i}_{j}")

    # Constraints to link auxiliary host_utilizations with the absolute difference of pairs
    for i in range(n):
        for j in range(i + 1, n):
            m.addConstr(differences[i, j] >= host_utilizations[i] - host_utilizations[j], f"DiffPos_{i}_{j}")
            m.addConstr(differences[i, j] >= host_utilizations[j] - host_utilizations[i], f"DiffNeg_{i}_{j}")
    
    # Objective: Minimize the sum of all absolute differences
    m.setObjectiveN(gp.quicksum(differences[i, j] for i in range(n) for j in range(i + 1, n)), index=1, priority=1)

    # ========== Minimize distance between current and prev weights ============
    
    # do this only if you have all the previous weights
    was_previous_the_same_topology = all(worker.name in previous_w for worker in _workers) and len(previous_w) == len(_workers)
    
    if was_previous_the_same_topology:
        
        print("Same topology;", "doing the distance optimization")
        
        n = len(_workers)
        abs_diff = m.addVars(n, vtype=GRB.CONTINUOUS, name="abs_diff")
        
        # set the new objective to minimize the distance between the weights
        for i, worker in enumerate(_workers):
            m.addConstr(abs_diff[i] >= w[worker.name] - previous_w[worker.name])
            m.addConstr(abs_diff[i] >= previous_w[worker.name] - w[worker.name])
        
        m.setObjectiveN(gp.quicksum(abs_diff[i] for i in range(n)), index=2, priority=0)
        
    else:
        
        print("Different topology;", "adding distance objective with 0 weight") 
        
        n = len(_workers)
        abs_diff = m.addVars(n, vtype=GRB.CONTINUOUS, name="abs_diff")
        
        # set the new objective to minimize the distance between the weights
        for i, worker in enumerate(_workers):
            m.addConstr(abs_diff[i] >= w[worker.name] - 0.0)
            m.addConstr(abs_diff[i] >= 0.0 - w[worker.name])
        
        m.setObjectiveN(gp.quicksum(abs_diff[i] for i in range(n)), index=2, priority=0)
    
    # ============================== Optimize! =================================
    
    m.optimize()
    
    # =========================== Done Optimization ============================
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        print(vars)
        
    if m.Status == GRB.OPTIMAL:        
        
        vars = {v.varName: v.x for v in m.getVars()}
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
            else:
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        # set the previous weights to the current weights
        previous_w = {worker.name: vars[f"w_{worker.name}"] for worker in _workers}
        print("New previous weights:", previous_w)            
        
        print(to_return)
        
        return to_return, m, t_min, t_consumed

    else:
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = 0.0
            else:
                results[worker.tenant][worker.name] = 0.0
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        print(to_return)
        
        return to_return, m, t_min, t_consumed
        
def run_generic_model(
    _hosts: List[Host],
    _tenants: List[Tenant],
    _workers: List[Worker]) -> str:
    
    global previous_w
    
    # =========================== Begin Optimization ===========================
    
    # MIP  model formulation
    m = gp.Model("lb")
    
    #  ============================= Set Variables =============================
    
    # set host capacity for each host
    cap = {}
    for h in _hosts:
        cap[h.name] = m.addVar(lb=h.cap, ub=h.cap, vtype=GRB.CONTINUOUS,
                        name=f"cap_{h.name}")
    
    # # set variables for the tenant loads
    # t = {}
    # for tenant in _tenants:
    #     t[tenant.name] = m.addVar(lb=tenant.load, ub=tenant.load, 
    #                               vtype=GRB.CONTINUOUS, name=f"t_{tenant.name}")
    
    # set variables for the workers
    w = {}   
    for worker in _workers:
        w[worker.name] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                           name=f"w_{worker.name}")
    
    print(w)

    t_min = {}
    for tenant in _tenants:
        t_min_value = min(tenant.fshareload, tenant.load)
        t_min[tenant.name] = m.addVar(lb=t_min_value, ub=t_min_value,
                                      vtype=GRB.CONTINUOUS,
                                 name=f"t_min_{tenant.name}")
        
    t_consumed = {}
    for tenant in _tenants:
        print(f"{tenant.name}: (lb: {min(tenant.fshareload, tenant.load)}, ub: {tenant.load})")
        # t_consumed[tenant.name] = m.addVar(lb=0.0,
        #                                    vtype=GRB.CONTINUOUS,
        #                          name=f"t_consumed_{tenant.name}")
        # We could also add the constraints here instead of adding them later, maybe that saves time in optimization
        t_consumed[tenant.name] = m.addVar(lb=min(tenant.fshareload, tenant.load), 
                                           ub=tenant.load,
                                           vtype=GRB.CONTINUOUS,
                                 name=f"t_consumed_{tenant.name}")
    
    t_excess_consumed = {}
    for tenant in _tenants:
        t_excess_consumed[tenant.name] = m.addVar(lb=0.0, vtype=GRB.CONTINUOUS,
                                 name=f"t_excess_consumed_{tenant.name}")
    
    t_log_excess_consumed = {}
    for tenant in _tenants:
        t_log_excess_consumed[tenant.name] = m.addVar(vtype=GRB.CONTINUOUS,
                                                      lb=-GRB.INFINITY,
                                                      ub=GRB.INFINITY,
                                 name=f"t_log_excess_consumed_{tenant.name}")
    
    one = m.addVar(lb=1.0, ub=1.0, vtype=GRB.CONTINUOUS, name="one")
    
    # ======================= Set Utilization Objective ========================
    
    sum_fshareloads = sum(tenant.fshareload for tenant in _tenants)
    
    weighted_sum_of_t_excess_consumption = gp.quicksum(
        ((tenant.fshareload / sum_fshareloads) * t_log_excess_consumed[tenant.name]
         for tenant in _tenants))
    
    # sum_workers = gp.quicksum((w[worker.name] for worker in _workers))
         
    m.setObjective(weighted_sum_of_t_excess_consumption, GRB.MAXIMIZE)
    
    # ============================ Set Constraints =============================
    
    # for each tenant, set t_log_excess_consumed = log(t_consumed)
    for tenant in _tenants:
        m.addGenConstrLog(t_excess_consumed[tenant.name],
                          t_log_excess_consumed[tenant.name])
    
    # for each tenant, set t_excess_consumed = t_consumed - t_min
    for tenant in _tenants:
        m.addConstr(
            t_excess_consumed[tenant.name] == 
            t_consumed[tenant.name] - t_min[tenant.name],
            name=f"t_excess_consumed_{tenant.name}")
    
    # at each h, sum(w ∈ h) <= cap
    for host in _hosts:
        m.addConstr(gp.quicksum(
            (w[worker.name] for worker in _workers if worker.host == host.name)) 
                    <= cap[host.name],
                    name=f"h_{host.name}")
        
    # for each tenant t, set t_consumed = sum(w ∈ t)
    for tenant in _tenants:
        m.addConstr(
            t_consumed[tenant.name] == gp.quicksum(
                (w[worker.name]
                for worker in _workers if worker.tenant == tenant.name)),
            name=f"t_consumed_{tenant.name}")
        
    # # at each t, t_consumed <= t
    # for tenant in _tenants:
    #     m.addConstr(
    #         t_consumed[tenant.name] <= tenant.load,
    #         name=f"t_upper_{tenant.name}")
    
    # # for each tenant, t_consumed >= t_min
    # for tenant in _tenants:
    #     m.addConstr(
    #         t_consumed[tenant.name] >= min(tenant.fshareload, tenant.load),
    #         name=f"t_lower_{tenant.name}")
    
    # ============================== Optimize! =================================
    
    m.optimize()
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        print(vars)
    
        # ==================== Variance Objective Optimization =====================
        
        t_consumed_min = {}
        for tenant in _tenants:
            t_consumed_min[tenant.name] = m.addVar(
                lb=vars[f"t_consumed_{tenant.name}"], 
                ub=vars[f"t_consumed_{tenant.name}"], 
                vtype=GRB.CONTINUOUS,
                name=f"t_consumed_min_{tenant.name}")
            
        # for each tenant, t_consumed_min <= t_consumed
        for tenant in _tenants:
            m.addConstr(
                t_consumed_min[tenant.name] <= t_consumed[tenant.name],
                name=f"t_consumed_min_{tenant.name}")
        
        # set spare caps
        sp = {}
        for host in _hosts:
            sp[host.name] = m.addVar(vtype=GRB.CONTINUOUS, name=f"sp_{host.name}")
            
        # for each host, sp = (cap - sum(w))
        for host in _hosts:
            m.addConstr(
                sp[host.name] == (cap[host.name] - gp.quicksum(
                    (w[worker.name] 
                    for worker in _workers if worker.host == host.name))),
                name=f"sp_{host.name}")
            
        # Compute the mean of spare caps
        mean = (1 / len(sp)) * gp.quicksum((sp[host.name] for host in _hosts))

        # Add the variance objective: minimize (1/n) * sum((x_i - mean)^2)
        variance = (1 / len(sp)) * gp.quicksum(
            (sp[host.name] - mean) * (sp[host.name] - mean) for host in _hosts)
            
        m.setObjective(variance, GRB.MINIMIZE)
        
        m.optimize()
        
    
        if m.Status == GRB.OPTIMAL:
            
            # =========================== Optimization minimize distanc between weights ============================
            
            # do this only if you have all the previous weights
            was_previous_the_same_topology = all(worker.name in previous_w for worker in _workers) and len(previous_w) == len(_workers)
            
            if was_previous_the_same_topology:
                
                print("Same topology;", "doing the distance optimization")
            
                # set that the variance objective is no more as much as what it was in the last optimization
                max_var = m.ObjVal
                m.addConstr(variance <= max_var)
                
                n = len(_workers)
                abs_diff = m.addVars(n, vtype=GRB.CONTINUOUS, name="abs_diff")
                
                # set the new objective to minimize the distance between the weights
                for i, worker in enumerate(_workers):
                    m.addConstr(abs_diff[i] >= w[worker.name] - previous_w[worker.name])
                    m.addConstr(abs_diff[i] >= previous_w[worker.name] - w[worker.name])
                
                m.setObjective(gp.quicksum(abs_diff[i] for i in range(n)), GRB.MINIMIZE)
                
                m.optimize()
                
                print("Same topology;", "did the distance optimization")
                
            else:
                
                print("Different topology;", "adding distance objective with 0 weight") 
                
                # set that the variance objective is no more as much as what it was in the last optimization
                max_var = m.ObjVal
                m.addConstr(variance <= max_var)
                
                n = len(_workers)
                abs_diff = m.addVars(n, vtype=GRB.CONTINUOUS, name="abs_diff")
                
                # set the new objective to minimize the distance between the weights
                for i, worker in enumerate(_workers):
                    m.addConstr(abs_diff[i] >= w[worker.name] - 0.0)
                    m.addConstr(abs_diff[i] >= 0.0 - w[worker.name])
                
                m.setObjective(gp.quicksum(abs_diff[i] for i in range(n)), GRB.MINIMIZE)
                
                m.optimize()
                
                print("Different topology;", "did the distance optimization")
    
    # =========================== Done Optimization ============================
    
    if m.Status == GRB.OPTIMAL:
        vars = {v.varName: v.x for v in m.getVars()}
        print(vars)
        
    if m.Status == GRB.OPTIMAL:        
        
        vars = {v.varName: v.x for v in m.getVars()}
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
            else:
                results[worker.tenant][worker.name] = vars[f"w_{worker.name}"]
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        # set the previous weights to the current weights
        previous_w = {worker.name: vars[f"w_{worker.name}"] for worker in _workers}
        print("New previous weights:", previous_w)            
        
        print(to_return)
        
        return to_return

    else:
        
        results = {}
        for worker in _workers:
            if worker.tenant not in results:
                results[worker.tenant] = {}
                results[worker.tenant][worker.name] = 0.0
            else:
                results[worker.tenant][worker.name] = 0.0
        to_return = {
            "status": m.Status,
            "result": results
        }
        
        print(to_return)
        
        return to_return
    
# run generic model from json input (from cc)
def run_from_json(hosts, tenants, workers):
    hosts = [Host(h["name"], h["cap"]) for h in hosts]
    tenants = [Tenant(t["name"], t["load"], t["fshareload"]) for t in tenants]
    workers = [Worker(w["name"], w["tenant"], w["host"]) for w in workers]
    return run_generic_model(hosts, tenants, workers)
    
# test run for the 3-node scenario on the newly written generic model func
def test_3_node_run_generic_model(host_cap, tenant_loads):
    
    hosts = [
        Host("0", host_cap),
        Host("1", host_cap),
        Host("2", host_cap)
    ]
    tenants = [
        Tenant("0", tenant_loads[0], 0.5 * host_cap),
        Tenant("1", tenant_loads[1], 1.5 * host_cap),
        Tenant("2", tenant_loads[2], 1.5 * host_cap)
    ]
    workers = [
        Worker("00", "0", "0"),
        Worker("10", "1", "0"),
        Worker("11", "1", "1"),
        Worker("21", "2", "1"),
        Worker("22", "2", "2"),        
    ]
    
    return run_generic_model(hosts, tenants, workers)
    
app = Flask(__name__)

@app.route('/', methods=['GET', 'POST'])
def gurobi_server():
    
    print("======================reached here")
    
    if request.method == "GET":
    
        print("reached here", request.args, request.args["host_cap"])
        start_time = time()
        
        variables = run_general_model(
            float(request.args["host_cap"]), 
            [float(request.args["t2"]),
            float(request.args["t0"]),
            float(request.args["t1"])])
       
        time_taken = time() - start_time
        print(f"{time_taken*1000:.2f} ms")
        
        return dumps(variables)
        
    elif request.method == "POST":
        
        start_time = time()
        print("reached here")
        request_data = request.get_json(force=False)
        print("Received:", request_data)
        hosts, tenants, workers = request_data[0], request_data[1], request_data[2]
        
        variables = run_from_json(hosts, tenants, workers)
        
        time_taken = time() - start_time
        print(f"{time_taken*1000:.2f} ms")
        
        return dumps(variables)  

@app.route('/reset', methods=['GET'])
def reset_weights():
    
    global previous_w
    previous_w = {}
    
    return "Weights reset!"

import sys

import logging
# Configure logging
logging.basicConfig(level=logging.DEBUG)
logger = logging.getLogger(__name__)

if __name__ == '__main__':
    
    if len(sys.argv) > 1:
        if sys.argv[1] == "-f":
            
            filename = sys.argv[2]
            
            with open(filename, "r") as f:
                input_json = f.read()
            
            start_time = time()
            input = json.loads(input_json)
            print("Input:", input)
            
            hosts, tenants, workers = input[0], input[1], input[2]
            output = run_from_json(hosts, tenants, workers)
            
            time_taken = time() - start_time
            print(f"{time_taken*1000:.2f} ms")
            
            output_json = dumps(output)
            
            with open(filename + "_output", "w") as f:
                f.write(output_json)
              
        else:
            print("Invalid argument, use -f to run sample json")
            
    else:
        print("======================reached here")
        app.run(host="localhost", port=5000, debug=True)
    
    # run_general_model(210, [79.8, 119, 202])
    # b = test_3_node_run_generic_model(200, [100, 300, 200])
    
    # print(a)
    # print(b)
    
    # # run_general_model(2, [89/100, 236/100, 185/100])
    # result = run_general_model(float(sys.argv[1]), [float(v) for v in sys.argv[2:]])
    
    # print()
    
    # print(
    #     {
    #         "node1-app3": f'{result["t20"]:.2f}',
    #         "node1-app1": f'{result["t00"]:.2f}',
    #         "node2-app1": f'{result["t01"]:.2f}',
    #         "node2-app2": f'{result["t11"]:.2f}',
    #         "node3-app2": f'{result["t12"]:.2f}'
    #     }
    # )

# run_model(20, 100, 30, 10)

# times = {}

# for size in [1, 10, 100, 1000, 10000, 100000]:

#     a = np.random.randint(0, 30, size=size)

#     start_time = time()
#     run_general_model(20, a)
#     time_taken = time() - start_time
#     print(f"{time_taken*1000:.2f} ms")

#     a = np.random.randint(0, 30, size=size)

#     start_time = time()
#     run_general_model(20, a)
#     time_taken = time() - start_time
#     print(f"{time_taken*1000:.2f} ms")
    
#     times[size] = time_taken*1000
    
# print(times)

# run_general_model(20, [10, 100, 40])

In [ ]:
from typing import List, Tuple
import time
import numpy as np


N_WORKERS_EXPONENTIAL_DISTR_LAMBDA = 17
WORKER_LOAD_EXPONENTIAL_DISTR_LAMBDA = 0.075 #0.7
HOST_CAPACITY = 1.0

class Host:
    def __init__(self, name: str, cap: float):
        self.name: str = name
        self.cap: str = cap
        self.worker_ids: List[int] = []
        
    def __str__(self):
        return f"{self.name}: cap={self.cap}, n_workers={len(self.worker_ids)}"
        
class Tenant:
    def __init__(self, name: str, load: float, fshare: float = 0.0):
        self.name: str = name
        self.load: float = load
        self.fshareload: float = fshare
        
    def __str__(self):
        return f"{self.name}: load={self.load}, fshareload={self.fshareload}"

class Worker:
    def __init__(self, name: str, tenant: str, host: str, tenant_id: int = -1):
        self.name: str = name
        self.tenant: str = tenant
        self.tenant_id: int = tenant_id
        self.host: str = host
        
    def __str__(self):
        return f"{self.name}: tenant={self.tenant}, host={self.host}"

def get_topology(n_hosts: int) -> Tuple[List[Host], List[Tenant], List[Worker]]:
    
    print(f"number of hosts: {n_hosts}")
    
    n_tenants = int(n_hosts * 0.65)
    print(f"number of tenants: {n_tenants}")    
    
    n_workers_per_ms = list(map(int, np.random.exponential(
        N_WORKERS_EXPONENTIAL_DISTR_LAMBDA, size=n_tenants)))
    print(f"number of workers: {np.sum(n_workers_per_ms)}")
    
    # ensure all n_workers_per_ms are at least 1
    n_workers_per_ms = [max(1, n) for n in n_workers_per_ms]
    
    hosts = [Host(f"host{i}", HOST_CAPACITY) for i in range(n_hosts)]
    
    tenants = []
    workers = []
    worker_id = 0
    for i in range(n_tenants):
        
        tenant_load = np.sum(np.random.exponential(
            WORKER_LOAD_EXPONENTIAL_DISTR_LAMBDA, size=n_workers_per_ms[i]))
        ``
        tenant = Tenant(f"tenant{i}", tenant_load)
        tenants.append(tenant)
        
        for j in range(n_workers_per_ms[i]):
            
            host_idx = np.random.randint(0, n_hosts)
            host_name = f"host{host_idx}"
            
            worker = Worker(f"tenant{i}_{j}", tenant.name, host_name, i)
            workers.append(worker)
            
            hosts[host_idx].worker_ids.append(worker_id)
            
            worker_id += 1
    
    for host in hosts:
        
        fshare_of_each_worker = HOST_CAPACITY / len(host.worker_ids) if len(host.worker_ids) > 0 else 0.0
        
        for worker_id in host.worker_ids:
            worker = workers[worker_id]
            tenant = tenants[worker.tenant_id]
            tenant.fshareload += fshare_of_each_worker
    
    # print("Hosts:")
    # for host in hosts:
    #     print(host)
    
    # print("Tenants:")
    # for tenant in tenants:
    #     print(tenant)
        
    # print("Workers:")
    # for worker in workers:
    #     print(worker)
    
    return hosts, tenants, workers, n_workers_per_ms

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

data = []
for n_hosts in [1, 3, 5, 10, 100, 1000, 10000]:
    hosts, tenants, workers, n_workers_per_ms = get_topology(n_hosts)
    data.append({
        "hosts": n_hosts,
        "total cap": sum(h.cap for h in hosts),
        "total load": sum(t.load for t in tenants)
    })

sns.lineplot(x="hosts", y="total cap", data=pd.DataFrame(data), label="Total Capacity", marker="o")
sns.lineplot(x="hosts", y="total load", data=pd.DataFrame(data), label="Total Load", marker="o")

# make both axes logarithmic
plt.yscale("log")
plt.xscale("log")

plt.show()

In [ ]:
import pandas as pd

def run_scale_experiment(n_hosts):
    
    data = []
    
    for n_hosts in n_hosts: #, 125, 250, 500, 1000, 2000, 4000, 8000, 16000, 32000, 64000]:
        hosts, tenants, workers, n_workers_per_ms = get_topology(n_hosts)
        
        print("===========================================================")
        print(f"Topology: {n_hosts} hosts, {len(tenants)} tenants, {len(workers)} workers")
        print("=================Quadratic Model===========================")
        start_time = time.time()
        
        result = run_generic_model(hosts, tenants, workers)
        
        time_taken = (time.time() - start_time)*1000
        print("Solved Quadratic Model in ", time_taken)
        data.append(["quadratic", n_hosts, len(tenants), len(workers), time_taken, [t.load for t in tenants], result])
        
        print("===========================================================")
        print(f"Topology: {n_hosts} hosts, {len(tenants)} tenants, {len(workers)} workers")
        print("=================Linear Model===========================")
        start_time = time.time()
        
        result, m, t_min, t_consumed = run_generic_linear_model(hosts, tenants, workers)
        
        time_taken = (time.time() - start_time)*1000
        print("Solved Linear Model in ", time_taken)
        data.append(["linear", n_hosts, len(tenants), len(workers), time_taken, [t.load for t in tenants], result])
        
        print("===========================================================")
        print(f"Topology: {n_hosts} hosts, {len(tenants)} tenants, {len(workers)} workers")
        print("=================Rerun Linear Model===========================")
        
        print("prev loads:", [t.load for t in tenants])
        # change tenant loads
        for i in range(len(tenants)):
            tenants[i].load = np.sum(np.random.exponential(
                WORKER_LOAD_EXPONENTIAL_DISTR_LAMBDA, size=n_workers_per_ms[i]))
        print("new tenant loads:", [t.load for t in tenants])
        
        start_time = time.time()
        
        result, m, t_min, t_consumed = rerun_generic_linear_model(m, t_min, t_consumed, hosts, tenants, workers)
        
        time_taken = (time.time() - start_time)*1000
        print("Solved rerunLinear Model in ", time_taken)
        data.append(["rerun-linear", n_hosts, len(tenants), len(workers), time_taken, [t.load for t in tenants], result])
        
    return data

df = pd.DataFrame(columns=['variation', 'n_hosts', 'n_tenants', 'n_workers', 'time_taken', 'tenant_loads', 'result'])

# df = pd.concat([df, pd.DataFrame(run_scale_experiment(), columns=['n_hosts', 'time_taken'])])
# df

In [ ]:
df

In [ ]:
for _ in range(1, 10):
    for n_hosts in [1, 3, 5, 10, 100, 150, 200, 250, 300, 350]:
        data = run_scale_experiment(n_hosts)
        with open("scale_experiment_log.json", "a") as f:
            for run in data:
                f.write(dumps({
                    "variation": run[0],
                    "n_hosts": run[1],
                    "n_tenants": run[2],
                    "n_workers": run[3],
                    "time_taken": run[4],
                    "tenant_loads": run[5],
                    "result": run[6]
                }) + "\n")
        # df = pd.concat([df, pd.DataFrame(data, columns=['variation', 'n_hosts', 'n_tenants', 'n_workers', 'time_taken', 'tenant_loads', 'result'])])

In [ ]:
data = run_scale_experiment([10])

In [ ]:
pd.DataFrame(data, columns=['variation', 'n_hosts', 'n_tenants', 'n_workers', 'time_taken', 'result'])

In [ ]:
data[0][5]

In [ ]:
data[1][5]

In [ ]:
data[2][5]

In [ ]:
data[2][5] == data[1][5]

In [ ]:
all(data[0][5] == d[ ])

In [ ]:
{"a": ["4", 2, 3], "b": [4, 5, 6]} == {"a": ["5", 2, 3], "b": [4, 5, 6]} 

In [ ]:
for _ in range(1):
    n_hosts = [3, 5, 10, 25, 50, 100, 150, 200, 250, 300, 350]
    df = pd.concat([df, pd.DataFrame(run_scale_experiment(n_hosts), columns=['variation', 'n_hosts', 'n_tenants', 'n_workers', 'time_taken'])])

df

In [ ]:
for _ in range(1):
    for variation in ["quadratic", "linear", "rerun-linear"]:
        n_hosts = [300]
        df = pd.concat([df, pd.DataFrame(run_scale_experiment(n_hosts, variation), columns=['variation', 'n_hosts', 'n_tenants', 'n_workers', 'time_taken'])])

df

In [ ]:
n_hosts = [800]
df = pd.concat([df, pd.DataFrame(run_scale_experiment(n_hosts), columns=['n_hosts', 'time_taken'])])
df

In [ ]:
n_hosts = [3, 5]
df = pd.concat([df, pd.DataFrame(run_scale_experiment(n_hosts), columns=['n_hosts', 'n_tenants', 'n_workers', 'time_taken'])])
df

In [ ]:
df.groupby('n_hosts').mean()

In [ ]:
# plot the results of the experiment x axis would be the number of hosts and y axis would be the time taken to solve the problem

import matplotlib.pyplot as plt
import seaborn as sns
# have an exponential y axis
sns.lineplot(data=df, x='n_hosts', y='time_taken', marker='o')
plt.yscale('log')
plt.xlabel("Number of hosts")
plt.ylabel("Time taken (ms)")
plt.title("Time taken to solve the problem for different number of hosts")

plt.show()


In [ ]:
# plot the results of the experiment x axis would be the number of hosts and y axis would be the time taken to solve the problem


# have an exponential y axis
sns.lineplot(data=df, x='n_hosts', y='time_taken', marker='o', hue='variation')
plt.yscale('log')
plt.xlabel("Number of hosts")
plt.ylabel("Time taken (ms)")
plt.title("Time taken to solve the problem for different number of hosts")

plt.show()


In [ ]:
df